# 02 — Construct the MD-derived Seitz table

This notebook uses only MD results plus the saved Argon scale.  It shows the raw
cavitation data, the homogeneous equation-of-state coverage used for interpolation,
the reduced-unit Seitz calculation, and its conversion to physical units.

REFPROP is deliberately not called here.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from md_Helpers import (
    ArgonLJScale, ProjectPaths, SQLiteRunDatabase,
    calculate_cavitation_seitz, cavitation_dataframe,
    query_seitz_eos_states,
)

MD_REPO = Path.home() / "MDsims"
DATA_DIRECTORY = MD_REPO / "Argon" / "data"
SCALE_PATH = DATA_DIRECTORY / "argon_lj_scale.json"
assert SCALE_PATH.exists(), "Run notebook 01 first."

scale_record = json.loads(SCALE_PATH.read_text())
scale = ArgonLJScale(
    epsilon_over_kb_K=scale_record["epsilon_over_kB_K"],
    sigma_nm=scale_record["sigma_nm"],
    mass_u=scale_record["mass_u"],
    epsilon_over_kb_uncertainty_K=scale_record["epsilon_over_kB_uncertainty_K"],
    sigma_uncertainty_nm=scale_record["sigma_uncertainty_nm"],
)
database = SQLiteRunDatabase(ProjectPaths().database)
database.initialize()
scale.summary()


## Raw cavitation and EOS selections


In [ ]:
CAVITATION_FILTERS = {
    "Nsteps": [500_000],
    "Phase_Separation_Status": "Separated",
}
EOS_FILTERS = {
    "Nsteps": 200_000,
    "dt": 0.002,
    "Ensemble": "NVT",
}

cavitations = cavitation_dataframe(database, **CAVITATION_FILTERS)
eos = query_seitz_eos_states(database, n_cells=45, **EOS_FILTERS)

print("Cavitation rows:", len(cavitations))
print("Cavitation temperatures:", sorted(cavitations["Therm_kT"].unique()))
print("EOS rows:", len(eos))
print("EOS temperatures:", sorted(eos["Therm_kT"].unique()))

cavitations[[
    "Run_ID", "N_Cells", "Therm_kT", "Initial_Density", "BoxLength",
    "rho_liquid", "rho_liquid_unc", "Pressure_Mean",
    "PE_Per_Particle_Mean", "PE_Per_Particle_SEM",
]]


In [ ]:
eos[[
    "Run_ID", "Therm_kT", "Density_End", "Pressure_Mean", "Pressure_SEM",
    "PE_Per_Particle_Mean", "PE_Per_Particle_SEM",
]].sort_values(["Therm_kT", "Density_End"])


## Visualize EOS coverage before interpolation


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
for temperature, group in eos.groupby("Therm_kT"):
    group = group.sort_values("Density_End")
    axes[0].errorbar(group["Density_End"], group["Pressure_Mean"], yerr=group["Pressure_SEM"],
                     marker="o", capsize=2, label=f"kT={temperature:g}")
    axes[1].errorbar(group["Density_End"], group["PE_Per_Particle_Mean"],
                     yerr=group["PE_Per_Particle_SEM"], marker="o", capsize=2,
                     label=f"kT={temperature:g}")

for temperature, group in cavitations.groupby("Therm_kT"):
    axes[0].scatter(group["rho_liquid"], np.zeros(len(group)), marker="x", s=70,
                    label=f"cavitation rho at kT={temperature:g}")

axes[0].set(xlabel="Reduced density", ylabel="Reduced pressure", title="Homogeneous EOS pressure coverage")
axes[1].set(xlabel="Reduced density", ylabel="Potential energy per particle", title="Homogeneous EOS energy coverage")
for axis in axes:
    axis.legend(fontsize=8)
    axis.grid(alpha=0.3)
plt.show()


## Calculate the MD Seitz threshold in reduced units


In [ ]:
results = calculate_cavitation_seitz(cavitations, eos)
reduced_columns = [
    "Run_ID", "Therm_kT", "Nc", "rho_c", "rho_liquid",
    "u_EOS", "P_EOS", "Q", "Q_uncertainty",
]
results[reduced_columns]


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
for temperature, group in results.groupby("Therm_kT"):
    group = group.sort_values("P_EOS")
    axes[0].errorbar(group["P_EOS"], group["Q"], yerr=group["Q_uncertainty"],
                     marker="o", capsize=3, label=f"kT={temperature:g}")
    axes[1].errorbar(group["rho_liquid"], group["Q"],
                     xerr=group["rho_liquid_uncertainty"], yerr=group["Q_uncertainty"],
                     marker="o", capsize=3, label=f"kT={temperature:g}")
axes[0].set(xlabel="EOS pressure (reduced)", ylabel="MD Seitz Q (reduced)", title="Reduced Q vs pressure")
axes[1].set(xlabel="Liquid density (reduced)", ylabel="MD Seitz Q (reduced)", title="Reduced Q vs liquid density")
for axis in axes:
    axis.legend()
    axis.grid(alpha=0.3)
plt.show()


## Convert the same states to physical Argon units


In [ ]:
results["T_K"] = scale.temperature(results["Therm_kT"])
results["T_C"] = results["T_K"] - 273.15
results["P_bar"] = scale.pressure(results["P_EOS"], "bar")
results["P_psia"] = results["P_bar"] * 14.5037738
results["Q_MD_keV"] = scale.energy(results["Q"], "keV")
results["Q_MD_uncertainty_keV"] = scale.energy_uncertainty(
    results["Q"], results["Q_uncertainty"], "keV"
)
results["rho_liquid_mol_L"] = scale.number_density(results["rho_liquid"], "mol/L")
results["rho_liquid_uncertainty_mol_L"] = scale.number_density_uncertainty(
    results["rho_liquid"], results["rho_liquid_uncertainty"], "mol/L"
)

physical_columns = [
    "Run_ID", "Therm_kT", "T_K", "P_EOS", "P_bar", "P_psia",
    "rho_liquid_mol_L", "Q", "Q_MD_keV", "Q_MD_uncertainty_keV",
]
results[physical_columns]


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
for temperature, group in results.groupby("T_K"):
    group = group.sort_values("P_bar")
    axes[0].errorbar(group["P_bar"], group["Q_MD_keV"], yerr=group["Q_MD_uncertainty_keV"],
                     marker="o", capsize=3, label=f"T={temperature:.2f} K")
    axes[1].errorbar(group["rho_liquid_mol_L"], group["Q_MD_keV"],
                     xerr=group["rho_liquid_uncertainty_mol_L"],
                     yerr=group["Q_MD_uncertainty_keV"], marker="o", capsize=3,
                     label=f"T={temperature:.2f} K")
axes[0].set(xlabel="Physical pressure (bar)", ylabel="MD Seitz Q (keV)", title="Physical Q vs pressure")
axes[1].set(xlabel="Liquid density (mol/L)", ylabel="MD Seitz Q (keV)", title="Physical Q vs density")
for axis in axes:
    axis.legend()
    axis.grid(alpha=0.3)
plt.show()


In [ ]:
output_path = DATA_DIRECTORY / "md_seitz_states.csv"
results.to_csv(output_path, index=False)
print("Saved:", output_path)
